In [21]:
# Block-1
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Import DS3000-Final Folder
import os
os.listdir('/content/drive/MyDrive/DS3000-Final')

import pandas as pd

# Loads the "ML-ready" charging-station dataset
df_stations = pd.read_csv('/content/drive/MyDrive/DS3000-Final/charging_station_ml.csv')
df_stations.head()




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,country_code,state_province,city,latitude,longitude,station_count,port_count,fast_station_count,fast_port_count,fast_station_share,fast_port_share,max_power_kw,median_power_kw,dc_fast_station_count,dc_ultra_station_count,has_fast_dc,has_ultra_dc
0,AD,Andorra la Vella,Andorra la Vella,42.506880,1.527992,2,4,0,0,0.0,0.0,22.0,22.0,0,0,0,0
1,AD,Andorra la Vella,Santa Coloma,42.496959,1.500966,1,2,0,0,0.0,0.0,22.0,22.0,0,0,0,0
2,AD,Canillo,Canillo,42.565901,1.598916,1,2,0,0,0.0,0.0,22.0,22.0,0,0,0,0
3,AD,Canillo,Soldeu,42.577997,1.663842,1,2,0,0,0.0,0.0,4.0,4.0,0,0,0,0
4,AD,Encamp,Encamp,42.535057,1.581334,2,3,0,0,0.0,0.0,22.0,14.7,0,0,0,0


In [22]:
# Block-2
# Overview of dataset columns, types, and missing values
df_stations.info()

# Basic statistics for numerical columns
df_stations.describe()

# Count of missing values per column
df_stations.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58642 entries, 0 to 58641
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country_code            58642 non-null  object 
 1   state_province          58642 non-null  object 
 2   city                    58642 non-null  object 
 3   latitude                58642 non-null  float64
 4   longitude               58642 non-null  float64
 5   station_count           58642 non-null  int64  
 6   port_count              58642 non-null  int64  
 7   fast_station_count      58642 non-null  int64  
 8   fast_port_count         58642 non-null  int64  
 9   fast_station_share      58642 non-null  float64
 10  fast_port_share         58638 non-null  float64
 11  max_power_kw            56902 non-null  float64
 12  median_power_kw         56902 non-null  float64
 13  dc_fast_station_count   58642 non-null  int64  
 14  dc_ultra_station_count  58642 non-null

,0
country_code,0
state_province,0
city,0
latitude,0
longitude,0
station_count,0
port_count,0
fast_station_count,0
fast_port_count,0
fast_station_share,0


In [23]:
# Block-3
# Filling errors/missing values in data-set
# Fill tiny missing 'fast_port_share' with median
df_stations['fast_port_share'].fillna(df_stations['fast_port_share'].median(), inplace=True)

# Fill 'max_power_kw' and 'median_power_kw' with median
df_stations['max_power_kw'].fillna(df_stations['max_power_kw'].median(), inplace=True)
df_stations['median_power_kw'].fillna(df_stations['median_power_kw'].median(), inplace=True)

# Verify missing values were handled
df_stations.isnull().sum()


/tmp/ipython-input-890859639.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_stations['fast_port_share'].fillna(df_stations['fast_port_share'].median(), inplace=True)
/tmp/ipython-input-890859639.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df

,0
country_code,0
state_province,0
city,0
latitude,0
longitude,0
station_count,0
port_count,0
fast_station_count,0
fast_port_count,0
fast_station_share,0


In [24]:
# Block-4
# Feature engineering for total power esitmates and Fast-DC ratio
# Total estimated station power
df_stations['total_station_power'] = df_stations['station_count'] * df_stations['median_power_kw']

# Fast DC ratio
df_stations['fast_dc_ratio'] = df_stations['dc_fast_station_count'] / df_stations['station_count']





In [25]:
# Block-5
# Loading & Cleaning EV_models data-set
df_ev = pd.read_csv('/content/drive/MyDrive/DS3000-Final/ev_models.csv')
df_ev.head()  # peek at first few rows
df_ev.info()  # check column types and missing values

# Mapping for battery size to vehicle size (Scale battery size with vehicle size)
body_battery_map = {
    'SUV': 75,
    'Sedan': 60,
    'Hatchback': 50,
    'Truck': 100
}

df_ev['battery_kWh_est'] = df_ev['body_style'].map(body_battery_map)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   make            128 non-null    object
 1   model           128 non-null    object
 2   market_regions  128 non-null    object
 3   powertrain      128 non-null    object
 4   first_year      128 non-null    int64 
 5   body_style      128 non-null    object
 6   origin_country  128 non-null    object
 7   age_2025        128 non-null    int64 
dtypes: int64(2), object(6)
memory usage: 8.1+ KB


In [26]:
# Block-6
# Clean / standardize body style values

# Define normalized category mapping
# Body-type generalizations
body_style_map = {
    'Sedan': 'Sedan',
    'Sedan/Wagon': 'Sedan',
    'Sedan/SUV': 'SUV',           # Hybrid category → choose dominant
    'Liftback': 'Sedan',          # Liftback typically sedan-like

    'Hatchback': 'Hatchback',
    'City car': 'Hatchback',      # Small EV city cars behave similarly

    'SUV': 'SUV',
    'SUV (3-row)': 'SUV',
    'SUV Coupe': 'SUV',
    'Crossover': 'Crossover',     # Keep separate for now
    'Pickup/SUV': 'Pickup',       # Mixed → treat as pickup

    'Pickup': 'Pickup',

    'Van': 'Van',
    'MPV': 'Van',                 # MPV = multi-purpose van

    'Coupe': 'Coupe',
    'Wagon': 'Wagon',
    'Shooting Brake': 'Wagon'     # Shooting brake = sporty wagon
}

# Apply mapping
df_ev['body_style_clean'] = df_ev['body_style'].map(body_style_map)

# Check results
df_ev['body_style_clean'].value_counts(dropna=False)


,count
body_style_clean,
SUV,68
Hatchback,21
Sedan,20
Crossover,7
Pickup,5
Wagon,3
Van,2
Coupe,2


In [27]:
# Block-7
# Create battery estimates by standardized body style

battery_mapping = {
    'Sedan': 65,
    'SUV': 85,
    'Crossover': 75,
    'Hatchback': 50,
    'Pickup': 110,
    'Van': 90,
    'Wagon': 70,
    'Coupe': 75
}

# Apply mapping
df_ev['battery_kWh_est'] = df_ev['body_style_clean'].map(battery_mapping)

# Fill any unmapped ones with median
df_ev['battery_kWh_est'].fillna(df_ev['battery_kWh_est'].median(), inplace=True)

# Verify no missing values
df_ev['battery_kWh_est'].isnull().sum()



/tmp/ipython-input-2453852421.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_ev['battery_kWh_est'].fillna(df_ev['battery_kWh_est'].median(), inplace=True)


np.int64(0)

In [28]:
# Block-8
# Load country + world summary datasets

df_country = pd.read_csv('/content/drive/MyDrive/DS3000-Final/country_summary.csv')
df_world = pd.read_csv('/content/drive/MyDrive/DS3000-Final/world_summary.csv')

df_country.head(), df_country.info(), df_world.head(), df_world.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country_code        121 non-null    object 
 1   country             121 non-null    object 
 2   station_count       121 non-null    int64  
 3   port_count          121 non-null    int64  
 4   fast_station_share  121 non-null    float64
 5   fast_port_share     121 non-null    float64
dtypes: float64(2), int64(2), object(2)
memory usage: 5.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country_code            121 non-null    object 
 1   country                 121 non-null    object 
 2   station_count           121 non-null    int64  
 3   port_count              121 non-null    int64  
 4   fast_station_c

(  country_code               country  station_count  port_count  \
 0           AD               Andorra             96         259   
 1           AE  United Arab Emirates            131         346   
 2           AF           Afghanistan              1           1   
 3           AL               Albania             15          16   
 4           AM               Armenia              4           6   
 
    fast_station_share  fast_port_share  
 0            0.062500         0.138996  
 1            0.175573         0.410405  
 2            0.000000         0.000000  
 3            0.600000         0.562500  
 4            0.250000         0.166667  ,
 None,
   country_code               country  station_count  port_count  \
 0           AD               Andorra             96         259   
 1           AE  United Arab Emirates            131         346   
 2           AF           Afghanistan              1           1   
 3           AL               Albania             15      

In [29]:
# Block-9
# Clean world_summary missing values + engineer features

# Fill missing power values with median value (safe because distribution is skewed)
df_world['max_power_kw'] = df_world['max_power_kw'].fillna(df_world['max_power_kw'].median())
df_world['median_power_kw'] = df_world['median_power_kw'].fillna(df_world['median_power_kw'].median())


# Feature Engineering
df_world['total_station_power'] = df_world['station_count'] * df_world['median_power_kw']
df_world['fast_dc_ratio'] = df_world['dc_fast_station_count'] / df_world['station_count']

# Verify
df_world.isnull().sum()


,0
country_code,0
country,0
station_count,0
port_count,0
fast_station_count,0
fast_port_count,0
fast_station_share,0
fast_port_share,0
max_power_kw,0
median_power_kw,0


In [30]:
# Block-10
#Load and inspect charging_station dataset for cleaning

df_station = pd.read_csv('/content/drive/MyDrive/DS3000-Final/charging_station.csv')

print("Shape:", df_station.shape)
df_station.head()
df_station.info()


Shape: (242417, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242417 entries, 0 to 242416
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              242417 non-null  int64  
 1   name            242417 non-null  object 
 2   city            242417 non-null  object 
 3   state_province  242417 non-null  object 
 4   country_code    242417 non-null  object 
 5   latitude        242417 non-null  float64
 6   longitude       242417 non-null  float64
 7   ports           242417 non-null  int64  
 8   power_kw        237757 non-null  float64
 9   power_class     242417 non-null  object 
 10  is_fast_dc      242417 non-null  bool   
dtypes: bool(1), float64(3), int64(2), object(5)
memory usage: 18.7+ MB


In [31]:
# Block-11
# Clean charging_station dataset (corrected version)

# 1. Fill missing power_kw using median power for each power_class
# Compute median power per class
class_medians = df_station.groupby('power_class')['power_kw'].median()

# Function to fill missing values using class median
def fill_power_kw(row):
    if pd.isna(row['power_kw']):
        cls = row['power_class']
        # If class exists in the median map, use it; otherwise fallback to global median
        return class_medians.get(cls, df_station['power_kw'].median())
    return row['power_kw']

df_station['power_kw'] = df_station.apply(fill_power_kw, axis=1)

# 2. Create power tier labels (same as before)
def classify_power(power):
    if power >= 150:
        return 'Ultra DC'
    elif power >= 50:
        return 'Fast DC'
    else:
        return 'Level 2'

df_station['power_tier'] = df_station['power_kw'].apply(classify_power)

# 3. Engineer total capacity feature
df_station['total_power_kw'] = df_station['power_kw'] * df_station['ports']

# Verify results
df_station[['power_kw', 'power_class', 'power_tier', 'ports', 'total_power_kw']].head()
df_station['power_tier'].value_counts()



,count
power_tier,
Level 2,191584
Fast DC,37329
Ultra DC,13504


In [32]:
# Block-12
# Merge station-level and world-level data-sets

# Ensure country_code fields match formatting
df_station['country_code'] = df_station['country_code'].str.upper()
df_world['country_code'] = df_world['country_code'].str.upper()

# Merge on country_code
df_merged = df_station.merge(
    df_world,
    on='country_code',
    how='left',
    suffixes=('_station', '_country')
)

print("Merged Shape:", df_merged.shape)
df_merged.head()


Merged Shape: (242417, 26)


,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,...,fast_station_count,fast_port_count,fast_station_share,fast_port_share,max_power_kw,median_power_kw,dc_fast_station_count,dc_ultra_station_count,total_station_power,fast_dc_ratio
0,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),...,6,36,0.0625,0.138996,300.0,22.0,6,2,2112.0,0.0625
1,301207,Parquing Costa Rodona,Encamp,UNKNOWN,AD,42.537213,1.727014,10,22.0,AC_HIGH_(22-49kW),...,6,36,0.0625,0.138996,300.0,22.0,6,2,2112.0,0.0625
2,301206,Hotel Naudi,Unknown City,UNKNOWN,AD,42.576811,1.666061,1,11.0,AC_L2_(7.5-21kW),...,6,36,0.0625,0.138996,300.0,22.0,6,2,2112.0,0.0625
3,301205,Hotel Piolets Soldeu Centre,Unknown City,UNKNOWN,AD,42.576466,1.667317,1,22.0,AC_HIGH_(22-49kW),...,6,36,0.0625,0.138996,300.0,22.0,6,2,2112.0,0.0625
4,301204,Hotel Serras,Unknown City,UNKNOWN,AD,42.579458,1.659215,3,11.0,AC_L2_(7.5-21kW),...,6,36,0.0625,0.138996,300.0,22.0,6,2,2112.0,0.0625


In [33]:
# Block-13
# Engineer relative infrastructure features

# Share of total national station power
df_merged['station_power_share'] = df_merged['total_power_kw'] / df_merged['total_station_power']

# Ports relative to national average
df_merged['avg_ports_per_station'] = df_merged['port_count'] / df_merged['station_count']
df_merged['ports_vs_country_avg'] = df_merged['ports'] - df_merged['avg_ports_per_station']

# Power percentile flag
df_merged['is_above_median_power'] = df_merged['power_kw'] > df_merged['median_power_kw']

# Fast DC vs country average deviation
df_merged['fast_dc_vs_country'] = df_merged['is_fast_dc'].astype(int) - df_merged['fast_dc_ratio']

df_merged.head()


,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,...,median_power_kw,dc_fast_station_count,dc_ultra_station_count,total_station_power,fast_dc_ratio,station_power_share,avg_ports_per_station,ports_vs_country_avg,is_above_median_power,fast_dc_vs_country
0,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),...,22.0,6,2,2112.0,0.0625,1.420455,2.697917,7.302083,True,0.9375
1,301207,Parquing Costa Rodona,Encamp,UNKNOWN,AD,42.537213,1.727014,10,22.0,AC_HIGH_(22-49kW),...,22.0,6,2,2112.0,0.0625,0.104167,2.697917,7.302083,False,-0.0625
2,301206,Hotel Naudi,Unknown City,UNKNOWN,AD,42.576811,1.666061,1,11.0,AC_L2_(7.5-21kW),...,22.0,6,2,2112.0,0.0625,0.005208,2.697917,-1.697917,False,-0.0625
3,301205,Hotel Piolets Soldeu Centre,Unknown City,UNKNOWN,AD,42.576466,1.667317,1,22.0,AC_HIGH_(22-49kW),...,22.0,6,2,2112.0,0.0625,0.010417,2.697917,-1.697917,False,-0.0625
4,301204,Hotel Serras,Unknown City,UNKNOWN,AD,42.579458,1.659215,3,11.0,AC_L2_(7.5-21kW),...,22.0,6,2,2112.0,0.0625,0.015625,2.697917,0.302083,False,-0.0625


In [34]:
# Block-14
# Inspect EV dataset for region fields (Mostly a double-check from earlier)

print("Shape:", df_ev.shape)
df_ev.info()
df_ev.head()
df_ev.columns


Shape: (128, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   make              128 non-null    object
 1   model             128 non-null    object
 2   market_regions    128 non-null    object
 3   powertrain        128 non-null    object
 4   first_year        128 non-null    int64 
 5   body_style        128 non-null    object
 6   origin_country    128 non-null    object
 7   age_2025          128 non-null    int64 
 8   battery_kWh_est   128 non-null    int64 
 9   body_style_clean  128 non-null    object
dtypes: int64(3), object(7)
memory usage: 10.1+ KB


Index(['make', 'model', 'market_regions', 'powertrain', 'first_year',
       'body_style', 'origin_country', 'age_2025', 'battery_kWh_est',
       'body_style_clean'],
      dtype='object')

In [35]:
# Block-15
# Prepare EV dataset for merging with main, cleaned data-set

# 1. Split multi-region entries into separate rows
df_ev_expanded = df_ev.assign(
    market_regions=df_ev['market_regions'].str.split(',')
).explode('market_regions')

# 2. Strip whitespace from region names
df_ev_expanded['market_regions'] = df_ev_expanded['market_regions'].str.strip()

# 3. Keep only necessary columns
df_ev_model = df_ev_expanded[['make', 'model', 'body_style_clean', 'battery_kWh_est', 'market_regions']]

# 4. Verify
print("Shape after expanding:", df_ev_model.shape)
df_ev_model.head()


Shape after expanding: (128, 5)


,make,model,body_style_clean,battery_kWh_est,market_regions
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME)
1,Tesla,Model 3,Sedan,65,Global (US/EU/UK/ME/CN)
2,Tesla,Model X,SUV,85,Global (US/EU/UK/ME)
3,Tesla,Model Y,SUV,85,Global (US/EU/UK/ME/CN)
4,Tesla,Cybertruck,Pickup,110,US (limited)


In [36]:
# Block-16
# Mapping global regions to countires
# Includes mapping EV models based on country/regional availablity to local regions
# Region mapping
region_to_country = {
    "US": ["US"],
    "EU": ["AD","AT","BE","BG","CH","CY","CZ","DE","DK","EE","ES","FI","FR","GB","GR","HU","IE","IT","LI","LT","LU","LV","MC","MT","NL","NO","PL","PT","RO","SE","SI","SK"],
    "UK": ["GB"],
    "ME": ["AL","BA","MK","RS","ME","XK","HR","GR","SI"],  # Example for “Middle Europe / Balkans”
    "CN": ["CN"],
    "Global (US/EU/UK/ME)": ["US","AD","AT","BE","BG","CH","CY","CZ","DE","DK","EE","ES","FI","FR","GB","GR","HU","IE","IT","LI","LT","LU","LV","MC","MT","NL","NO","PL","PT","RO","SE","SI","AL","BA","MK","RS","ME","XK","HR"],
    "Global (US/EU/UK/ME/CN)": ["US","AD","AT","BE","BG","CH","CY","CZ","DE","DK","EE","ES","FI","FR","GB","GR","HU","IE","IT","LI","LT","LU","LV","MC","MT","NL","NO","PL","PT","RO","SE","SI","AL","BA","MK","RS","ME","XK","HR","CN"],
    "US (limited)": ["US"]
}
# Map EV rows to country codes
df_ev_model['country_codes'] = df_ev_model['market_regions'].map(region_to_country)

# Explode so each EV-country combination is its own row
df_ev_country = df_ev_model.explode('country_codes').rename(columns={'country_codes':'country_code'})

# Verify
print("Shape after mapping to countries:", df_ev_country.shape)
df_ev_country.head()


Shape after mapping to countries: (282, 6)


/tmp/ipython-input-2575282071.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ev_model['country_codes'] = df_ev_model['market_regions'].map(region_to_country)


,make,model,body_style_clean,battery_kWh_est,market_regions,country_code
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME),US
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME),AD
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME),AT
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME),BE
0,Tesla,Model S,Sedan,65,Global (US/EU/UK/ME),BG


In [37]:
# Block-17
# Merge stations data-set with EVs by country data-set
df_station_ev = df_station.merge(
    df_ev_country,
    on='country_code',
    how='left'   # keep all stations, even if some have no EVs techincally
)

# Verify the merge
print("Shape after merging stations with EVs:", df_station_ev.shape)
df_station_ev.head()


Shape after merging stations with EVs: (1448964, 18)


,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,is_fast_dc,power_tier,total_power_kw,make,model,body_style_clean,battery_kWh_est,market_regions
0,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),True,Ultra DC,3000.0,Tesla,Model S,Sedan,65.0,Global (US/EU/UK/ME)
1,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),True,Ultra DC,3000.0,Tesla,Model 3,Sedan,65.0,Global (US/EU/UK/ME/CN)
2,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),True,Ultra DC,3000.0,Tesla,Model X,SUV,85.0,Global (US/EU/UK/ME)
3,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),True,Ultra DC,3000.0,Tesla,Model Y,SUV,85.0,Global (US/EU/UK/ME/CN)
4,301207,Parquing Costa Rodona,Encamp,UNKNOWN,AD,42.537213,1.727014,10,22.0,AC_HIGH_(22-49kW),False,Level 2,220.0,Tesla,Model S,Sedan,65.0,Global (US/EU/UK/ME)


In [38]:
# Block-18
# Use the correct merged DataFrame
# This assumes EV info per station with country exploded
# Normal if this takes a while to load
station_agg = df_station_ev.groupby('id').agg(
    total_evs = ('model', 'count'),
    avg_battery = ('battery_kWh_est', 'mean'),
    body_style_counts = ('body_style_clean', lambda x: x.value_counts().to_dict())
).reset_index()

# Expand the dictionary of body style counts into separate columns
body_style_df = station_agg['body_style_counts'].apply(pd.Series).fillna(0)

# Merge expanded body style counts back into main DataFrame
station_agg_final = pd.concat([station_agg.drop(columns='body_style_counts'), body_style_df], axis=1)

# Optional: convert counts to integers
station_agg_final[body_style_df.columns] = station_agg_final[body_style_df.columns].astype(int)

# Verify the final DataFrame
print("Shape after aggregating EV info per station:", station_agg_final.shape)
station_agg_final.head()


Shape after aggregating EV info per station: (242417, 7)


,id,total_evs,avg_battery,SUV,Pickup,Sedan,Van
0,2389,11,88.181818,6,3,2,0
1,2391,11,88.181818,6,3,2,0
2,2396,11,88.181818,6,3,2,0
3,2399,11,88.181818,6,3,2,0
4,2441,11,88.181818,6,3,2,0


In [39]:
# Block-19
#Final station summary with percentages and flags
# Merge back descriptive station info (name, city, state, etc.) into main data-set
station_summary = station_agg_final.merge(
    df_station[['id','name','city','state_province','country_code','latitude','longitude','ports','power_kw','power_class','is_fast_dc','power_tier','total_power_kw']],
    on='id',
    how='left'
)

# Reorder columns for clarity
cols_order = [
    'id','name','city','state_province','country_code','latitude','longitude',
    'ports','power_kw','power_class','is_fast_dc','power_tier','total_power_kw',
    'total_evs','avg_battery'
] + [col for col in station_agg_final.columns if col not in ['id','total_evs','avg_battery']]
station_summary = station_summary[cols_order]

# Compute percentage of each body style per station
body_style_cols = [col for col in station_agg_final.columns if col not in ['id','total_evs','avg_battery']]
for col in body_style_cols:
    station_summary[f'{col}_pct'] = station_summary[col] / station_summary['total_evs'] * 100

# Optional flags
station_summary['mostly_fast_dc'] = station_summary['is_fast_dc'] & (station_summary['total_evs'] >= 5)
station_summary['high_ev_count'] = station_summary['total_evs'] >= 10

# Verify final DataFrame
print("Shape of final station summary:", station_summary.shape)
station_summary.head()



Shape of final station summary: (242417, 25)


,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,...,SUV,Pickup,Sedan,Van,SUV_pct,Pickup_pct,Sedan_pct,Van_pct,mostly_fast_dc,high_ev_count
0,2389,Los Angeles Convention Center,Los Angeles,CA,US,34.040539,-118.271387,1,3.7,AC_L1_(<7.5kW),...,6,3,2,0,54.545455,27.272727,18.181818,0.0,False,True
1,2391,LADWP - John Ferraro Building,Los Angeles,CA,US,34.059133,-118.248589,1,50.0,DC_FAST_(50-149kW),...,6,3,2,0,54.545455,27.272727,18.181818,0.0,True,True
2,2396,State Capitol Parking Garage,Sacramento,CA,US,38.576769,-121.495022,1,3.7,AC_L1_(<7.5kW),...,6,3,2,0,54.545455,27.272727,18.181818,0.0,False,True
3,2399,CITYOFSANTAROSA,Santa Rosa,CA,US,38.438052,-122.711357,1,NaN,UNKNOWN,...,6,3,2,0,54.545455,27.272727,18.181818,0.0,False,True
4,2441,City of Pasadena - De Lacey Garage,Pasadena,CA,US,34.145138,-118.152655,1,3.7,AC_L1_(<7.5kW),...,6,3,2,0,54.545455,27.272727,18.181818,0.0,False,True


In [ ]:
# Block-20
# Interactive map-visualization of EV stations globally
# Be careful with keeping this output displayed, it takes up a significant amount of system resources, and I have had problems with saving the file while keeping displayed

import folium
from folium.plugins import MarkerCluster

# Create a map centered roughly at the geographic center of your dataset
map_center = [station_summary['latitude'].mean(), station_summary['longitude'].mean()]
ev_map = folium.Map(location=map_center, zoom_start=4)

# Cluster markers for readability
marker_cluster = MarkerCluster().add_to(ev_map)

# Choose a metric to color markers (e.g., total EVs)
for _, row in station_summary.iterrows():
    # Choose color based on total EVs
    if row['total_evs'] >= 20:
        color = 'darkred'
    elif row['total_evs'] >= 10:
        color = 'red'
    elif row['total_evs'] >= 5:
        color = 'orange'
    else:
        color = 'green'

    # Popup info
    popup_text = f"""
    <b>{row['name']}</b><br>
    {row['city']}, {row['state_province']}<br>
    Total EVs: {row['total_evs']}<br>
    Avg Battery: {row['avg_battery']:.1f} kWh<br>
    SUV: {row.get('SUV',0)}, Sedan: {row.get('Sedan',0)}, Pickup: {row.get('Pickup',0)}, Van: {row.get('Van',0)}
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5 + row['total_evs']*0.2,   # size proportional to total EVs
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(marker_cluster)

# Display the map
ev_map


In [ ]:
# Block-21
# Additional visulizations and graphics
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of total EVs per station
plt.figure(figsize=(10,6))
sns.histplot(station_summary['total_evs'], bins=30, kde=False, color='teal')
plt.title("Distribution of Total EVs per Station")
plt.xlabel("Total EVs")
plt.ylabel("Number of Stations")
plt.show()


plt.figure(figsize=(10,6))
sns.histplot(station_summary['avg_battery'], bins=30, kde=True, color='orange')
plt.title("Distribution of Average Battery Size per Station")
plt.xlabel("Average Battery (kWh)")
plt.ylabel("Number of Stations")
plt.show()

# Sum all body style counts
body_styles = ['SUV', 'Sedan', 'Pickup', 'Van']  # adjust based on your columns
body_style_totals = station_summary[body_styles].sum().sort_values(ascending=False)


plt.figure(figsize=(8,5))
sns.barplot(x=body_style_totals.index, y=body_style_totals.values, palette='muted')
plt.title("Total EVs by Body Style Across All Stations")
plt.ylabel("Number of EVs")
plt.show()


# Sum all body style counts
body_styles = ['SUV', 'Sedan', 'Pickup', 'Van']  # adjust based on your columns
body_style_totals = station_summary[body_styles].sum().sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=body_style_totals.index, y=body_style_totals.values, palette='muted')
plt.title("Total EVs by Body Style Across All Stations")
plt.ylabel("Number of EVs")
plt.show()


plt.figure(figsize=(10,6))
sns.boxplot(x='mostly_fast_dc', y='total_evs', data=station_summary)
plt.title("EV Counts by Station Type (Mostly Fast DC vs. Mostly AC)")
plt.xlabel("Mostly Fast DC")
plt.ylabel("Total EVs")
plt.show()


Key insights from cleaned data:

*   Most EV-charging stations average roughly 3.8 vechicles per station globally.
*   Most EV's have one of two standard battery sizes in (KWH).
*   EV-SUV's are a substantially more popular body style than Sedans (including coupes and hatches), Pickups or Vans.
*   Nearly all EV charging stations are comprised of majorly AC chargers, with a few Fast-DC or Ultra-DC ports.
*   Generally, larger stations (total kW), do support more EV's (higher EV-densities).

Data-Set Notes:
*   Relatively complete and comprehensive data-set (few missing or bad data entries).
*   Data is more limited from rural areas.
*   EV-Infrastructure varies greatly with country and geographical region.
*   Final cleaned data sets are: "station_summary.csv" & "station_agg_final.csv".
*   Most missing values were filled with median categorical values.
*   **SOME DATA-SETS AND GRAPHICS ARE SYSTEM-RESOURCE INTENSIVE AND MAY TAKE A WHILE TO LOAD/DISPLAY**

Notes for Model Creation:
*   Market for EV list can help give an idea of which EV's are being used in each region.
*   Total station power and number for ports, along with ratio of AC:Fast-DC can help give estimates of peak supply/capacity per station.
*   Very high correlation between battery capacity (kWh), body-style and charg-time (larger vehicles (mostly) mean larger batteries and longer charge times).
*   In map, darker coloured areas indicate higher EV-densities in the area around the station.













In [40]:
# Block-22: Export and Share

# Export final datasets
station_summary.to_csv('station_summary.csv', index=False)
station_agg_final.to_csv('station_agg_final.csv', index=False)

# Optional: Save key visualizations
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(station_summary['total_evs'], bins=30, kde=False, color='teal')
plt.title("Distribution of Total EVs per Station")
plt.xlabel("Total EVs")
plt.ylabel("Number of Stations")
plt.savefig('total_evs_distribution.png')
plt.close()

plt.figure(figsize=(10,6))
sns.histplot(station_summary['avg_battery'], bins=30, kde=True, color='orange')
plt.title("Distribution of Average Battery Size per Station")
plt.xlabel("Average Battery (kWh)")
plt.ylabel("Number of Stations")
plt.savefig('avg_battery_distribution.png')
plt.close()




In [42]:
# Block-23
# Prepare station_summary for modeling (categorical encoding)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Drop columns that should NOT be used for prediction
df_model = station_summary.drop(columns=['name'])

# 2. Identify categorical columns
categorical_cols = [
    'city',
    'state_province',
    'country_code',
    'power_class',
    'power_tier'
]

# 3. Identify numeric columns
numeric_cols = [col for col in df_model.columns if col not in categorical_cols]

# 4. Create OneHotEncoder pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'  # keep numeric features
)

# 5. Fit-transform the data
df_encoded = preprocessor.fit_transform(df_model)

# 6. Convert back to DataFrame
encoded_feature_names = (
    preprocessor.named_transformers_['cat']
    .get_feature_names_out(categorical_cols)
)

all_feature_names = list(encoded_feature_names) + numeric_cols

df_model_encoded = pd.DataFrame(df_encoded.toarray(), columns=all_feature_names)

# 7. Export ready-for-ML dataset
df_model_encoded.to_csv('station_summary_model_encoded.csv', index=False)

df_model_encoded.head()


KeyboardInterrupt: 